# 🛍️ Visual Search trên Shopee - Tuần 4
## Phương pháp: HỌC ĐA PHƯƠNG THỨC THỰC DỤNG
### Multimodal Late Fusion: EfficientNet-B4 + TF-IDF + FAISS

**Kiến trúc siêu vector:** EfficientNet-B4 (1792-D) + TF-IDF (3000-D) → Late Fusion → FAISS IndexFlatIP

---
> **Môi trường chạy:** Google Colab · GPU T4 · Python 3.10
> 
> **Dữ liệu:** Shopee - Price Match Guarantee (train.csv + train_images, ~34.250 ảnh)

In [ ]:
# ============================================================
# CELL 1: CÀI ĐẶT & IMPORT THƯ VIỆN
# ============================================================
!pip install faiss-gpu -q

import os
import re
import warnings
warnings.filterwarnings('ignore')

# --- Core ---
import numpy as np
import pandas as pd

# --- Deep Learning ---
import torch
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image, UnidentifiedImageError

# --- Machine Learning ---
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as sk_normalize

# --- FAISS ---
import faiss

# --- Visualization ---
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns

# --- Progress Bar ---
from tqdm.auto import tqdm

# --- Thiết lập chung ---
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Thiết bị đang dùng: {DEVICE}")
print(f"✅ PyTorch version: {torch.__version__}")
print(f"✅ FAISS version: {faiss.__version__}")
print(f"✅ GPU có sẵn: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# CELL 2: CẤU HÌNH ĐƯỜNG DẪN
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ─── Cấu hình đường dẫn gốc – chỉnh sửa nếu cần ───────────
DRIVE_ROOT  = '/content/drive/MyDrive'
PROJECT_DIR = os.path.join(DRIVE_ROOT, 'ShopeeSearch')   # Thư mục gốc dự án

# Đường dẫn dữ liệu
DATA_DIR   = os.path.join(PROJECT_DIR, 'data')            # Chứa train.csv, train_images/
TRAIN_CSV  = os.path.join(DATA_DIR, 'train.csv')
IMAGE_DIR  = os.path.join(DATA_DIR, 'train_images')

# Đường dẫn lưu features và kết quả
PROCESSED  = os.path.join(PROJECT_DIR, 'processed_features')  # Lưu file .npy, .csv trung gian
RESULTS    = os.path.join(PROJECT_DIR, 'results')             # Lưu metrics, ảnh biểu đồ

# Tự tạo thư mục nếu chưa tồn tại
for d in [DATA_DIR, PROCESSED, RESULTS]:
    os.makedirs(d, exist_ok=True)

# ─── Hyperparameters ───────────────────────────────────────
BATCH_SIZE    = 64
NUM_WORKERS   = 2
TFIDF_FEATS   = 3000
IMAGE_FEATS   = 1792
FUSED_DIM     = IMAGE_FEATS + TFIDF_FEATS   # 4792
K_LIST        = [1, 3, 5, 10]
MAX_K         = max(K_LIST)

print("✅ Cấu hình hoàn tất!")
print(f"   DATA_DIR  : {DATA_DIR}")
print(f"   PROCESSED : {PROCESSED}")
print(f"   RESULTS   : {RESULTS}")
print(f"   Siêu vector: {FUSED_DIM} chiều ({IMAGE_FEATS} + {TFIDF_FEATS})")

In [ ]:
# ============================================================
# CELL 3: LOAD & LỌC DỮ LIỆU CHUNG
# ============================================================
CANDIDATE_CSV = os.path.join(PROCESSED, 'candidate_df_chung.csv')

if os.path.exists(CANDIDATE_CSV):
    print(f"✅ Tải candidate_df_chung.csv từ cache: {CANDIDATE_CSV}")
    df = pd.read_csv(CANDIDATE_CSV)
else:
    print("📂 Load train.csv gốc...")
    raw_df = pd.read_csv(TRAIN_CSV)
    print(f"   Tổng số ảnh ban đầu: {len(raw_df):,}")

    # Chỉ giữ các label_group có từ 2 ảnh trở lên (để có positive pair)
    group_counts = raw_df['label_group'].value_counts()
    valid_groups = group_counts[group_counts >= 2].index
    df = raw_df[raw_df['label_group'].isin(valid_groups)].reset_index(drop=True)

    print(f"   Số ảnh sau lọc (label_group >= 2): {len(df):,}")
    print(f"   Số label_group hợp lệ: {df['label_group'].nunique():,}")

    # Lưu lại để dùng chung
    df.to_csv(CANDIDATE_CSV, index=False)
    print(f"   💾 Đã lưu: {CANDIDATE_CSV}")

# Đảm bảo cột image path tồn tại
df['image_path'] = df['image'].apply(lambda x: os.path.join(IMAGE_DIR, x))

# Tạo mapping: posting_id -> index và ngược lại
idx2posting = df['posting_id'].tolist()
posting2idx = {pid: i for i, pid in enumerate(idx2posting)}

# Ground-truth: mỗi posting_id → set các posting_id cùng label_group (trừ chính nó)
group2postings = df.groupby('label_group')['posting_id'].apply(set).to_dict()
gt_map = {
    row['posting_id']: group2postings[row['label_group']] - {row['posting_id']}
    for _, row in df.iterrows()
}

print(f"\n📊 DataFrame hiện tại: {len(df):,} ảnh | {df['label_group'].nunique():,} nhóm sản phẩm")
print(df[['posting_id', 'label_group', 'title', 'image']].head(3))

In [ ]:
# ============================================================
# CELL 4: TRÍCH XUẤT ĐẶC TRƯNG ẢNH – EfficientNet-B4
# ============================================================
EFFNET_NPY = os.path.join(PROCESSED, 'efficientnet_b4_features.npy')

# ── Dataset helper ─────────────────────────────────────────
class ShopeeImageDataset(Dataset):
    """Dataset đọc ảnh Shopee, trả về tensor đã transform."""

    # EfficientNet-B4 chuẩn: input 380×380
    transform = transforms.Compose([
        transforms.Resize(380),
        transforms.CenterCrop(380),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225]
        ),
    ])

    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = self.image_paths[idx]
        try:
            img = Image.open(path).convert('RGB')
            return self.transform(img), idx
        except (UnidentifiedImageError, FileNotFoundError, OSError):
            # Ảnh corrupt / thiếu → trả tensor zeros
            return torch.zeros(3, 380, 380), idx


# ── Load / trích xuất feature ───────────────────────────────
if os.path.exists(EFFNET_NPY):
    print(f"✅ Tải EfficientNet-B4 features từ cache: {EFFNET_NPY}")
    img_features = np.load(EFFNET_NPY)
else:
    print("🔧 Khởi tạo EfficientNet-B4 (pretrained)...")
    backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)
    # Bỏ lớp phân loại cuối, chỉ giữ feature extractor
    feature_extractor = nn.Sequential(
        backbone.features,
        backbone.avgpool,
        nn.Flatten()
    ).to(DEVICE).eval()

    dataset    = ShopeeImageDataset(df['image_path'].tolist())
    dataloader = DataLoader(dataset, batch_size=BATCH_SIZE,
                            num_workers=NUM_WORKERS, pin_memory=True)

    img_features = np.zeros((len(df), IMAGE_FEATS), dtype=np.float32)

    print(f"🚀 Trích xuất đặc trưng ảnh cho {len(df):,} ảnh...")
    with torch.no_grad():
        for images, indices in tqdm(dataloader, desc="EfficientNet-B4", unit="batch"):
            images = images.to(DEVICE)
            feats  = feature_extractor(images)   # (B, 1792)
            img_features[indices.numpy()] = feats.cpu().numpy()

    np.save(EFFNET_NPY, img_features)
    print(f"   💾 Đã lưu: {EFFNET_NPY}")

print(f"\n📐 img_features shape: {img_features.shape}")
print(f"   dtype : {img_features.dtype}")
print(f"   RAM   : {img_features.nbytes / 1e6:.1f} MB")

In [ ]:
# ============================================================
# CELL 5: TRÍCH XUẤT ĐẶC TRƯNG VĂN BẢN – TF-IDF
# ============================================================
TFIDF_NPY = os.path.join(PROCESSED, 'tfidf_features.npy')

def clean_text(text: str) -> str:
    """Làm sạch văn bản: bỏ ký tự đặc biệt, dấu câu, lowercase."""
    if not isinstance(text, str):
        return ''
    # Lowercase
    text = text.lower()
    # Bỏ ký tự không phải chữ/số/khoảng trắng (giữ lại cả ký tự unicode tiếng Việt)
    text = re.sub(r'[^\w\s]', ' ', text, flags=re.UNICODE)
    # Chuẩn hóa khoảng trắng thừa
    text = re.sub(r'\s+', ' ', text).strip()
    return text


if os.path.exists(TFIDF_NPY):
    print(f"✅ Tải TF-IDF features từ cache: {TFIDF_NPY}")
    txt_features = np.load(TFIDF_NPY)
else:
    print("🔧 Làm sạch văn bản (cột title)...")
    titles_clean = [
        clean_text(t)
        for t in tqdm(df['title'].tolist(), desc='clean_text', unit='doc')
    ]

    print(f"🔧 Fit TfidfVectorizer (max_features={TFIDF_FEATS})...")
    vectorizer = TfidfVectorizer(
        max_features=TFIDF_FEATS,
        sublinear_tf=True,          # log TF để giảm ảnh hưởng từ xuất hiện nhiều
        analyzer='word',
        token_pattern=r'\w+',
    )
    tfidf_sparse = vectorizer.fit_transform(titles_clean)  # sparse matrix

    # Chuyển sparse → dense → NumPy float32
    print("🔧 Chuyển sparse → dense...")
    txt_features = tfidf_sparse.toarray().astype(np.float32)  # (N, 3000)

    np.save(TFIDF_NPY, txt_features)
    print(f"   💾 Đã lưu: {TFIDF_NPY}")

print(f"\n📐 txt_features shape: {txt_features.shape}")
print(f"   dtype : {txt_features.dtype}")
print(f"   RAM   : {txt_features.nbytes / 1e6:.1f} MB")

# Kiểm tra mẫu
print(f"\n📝 Mẫu title gốc: {df['title'].iloc[0][:80]}")
print(f"   Sau clean    : {clean_text(df['title'].iloc[0])[:80]}")

In [ ]:
# ============================================================
# CELL 6: TÍCH HỢP LATE FUSION & FAISS
# ============================================================
FUSION_NPY = os.path.join(PROCESSED, 'fusion_eff_tfidf_features.npy')

# ── Bước 1: L2-Norm từng modality ──────────────────────────
print("🔧 Bước 1: L2-Normalize từng modality...")
img_normed = sk_normalize(img_features, norm='l2')   # (N, 1792)
txt_normed = sk_normalize(txt_features, norm='l2')   # (N, 3000)

# ── Bước 2: Concatenate → siêu vector ──────────────────────
print("🔧 Bước 2: Concatenate → siêu vector (4792-D)...")
fused = np.concatenate([img_normed, txt_normed], axis=1)  # (N, 4792)
assert fused.shape[1] == FUSED_DIM, f"Dimension mismatch: {fused.shape[1]} != {FUSED_DIM}"

# ── Bước 3: L2-Norm lần cuối trên siêu vector ──────────────
print("🔧 Bước 3: L2-Normalize siêu vector...")
fused = sk_normalize(fused, norm='l2').astype(np.float32)  # Đảm bảo float32 cho FAISS

np.save(FUSION_NPY, fused)
print(f"   💾 Đã lưu siêu vector: {FUSION_NPY}")
print(f"\n📐 fused shape: {fused.shape} | dtype: {fused.dtype}")

# ── Bước 4: Xây dựng FAISS Index ───────────────────────────
print("\n🚀 Xây dựng FAISS IndexFlatIP...")
index_cpu = faiss.IndexFlatIP(FUSED_DIM)   # Inner Product (≡ Cosine khi đã L2-norm)
index_cpu.add(fused)

# Đẩy lên GPU nếu có
if torch.cuda.is_available():
    res   = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(res, 0, index_cpu)
    print("   ✅ FAISS Index đã đưa lên GPU")
else:
    index = index_cpu
    print("   ⚠️  Chạy FAISS trên CPU")

print(f"   Tổng vector trong index: {index.ntotal:,}")

In [ ]:
# ============================================================
# CELL 7: TÍNH METRIC THỰC NGHIỆM
# ============================================================

def compute_ap_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Average Precision@K cho một query."""
    if not relevant_ids:
        return 0.0
    hits = 0
    score = 0.0
    for rank, pid in enumerate(retrieved_ids[:k], start=1):
        if pid in relevant_ids:
            hits += 1
            score += hits / rank
    return score / min(len(relevant_ids), k)


def compute_precision_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Precision@K cho một query."""
    if k == 0 or not relevant_ids:
        return 0.0
    hits = sum(1 for pid in retrieved_ids[:k] if pid in relevant_ids)
    return hits / k


def compute_recall_at_k(retrieved_ids: list, relevant_ids: set, k: int) -> float:
    """Recall@K cho một query."""
    if not relevant_ids:
        return 0.0
    hits = sum(1 for pid in retrieved_ids[:k] if pid in relevant_ids)
    return hits / len(relevant_ids)


# ── Tìm kiếm batch bằng FAISS ───────────────────────────────
SEARCH_K = MAX_K + 1   # +1 để loại bỏ chính ảnh query
N = len(df)

print(f"🔍 FAISS search {N:,} queries, Top-{SEARCH_K}...")
_, I = index.search(fused, SEARCH_K)   # I: (N, SEARCH_K) – indices trong index
print("   ✅ Tìm kiếm xong!")

# ── Tính metric ────────────────────────────────────────────
map5_list = []
precision_lists = {k: [] for k in K_LIST}
recall_lists    = {k: [] for k in K_LIST}
detail_rows     = []

print("📊 Tính metrics...")
for q_idx in tqdm(range(N), desc="Metrics", unit="query"):
    q_posting = idx2posting[q_idx]
    relevant  = gt_map[q_posting]   # set các posting cùng nhóm (không gồm query)

    # Lấy danh sách kết quả, loại bỏ chính query
    neighbor_indices = I[q_idx].tolist()
    retrieved = [
        idx2posting[ni]
        for ni in neighbor_indices
        if idx2posting[ni] != q_posting
    ][:MAX_K]

    # mAP@5
    ap5 = compute_ap_at_k(retrieved, relevant, k=5)
    map5_list.append(ap5)

    # Precision & Recall @K
    for k in K_LIST:
        precision_lists[k].append(compute_precision_at_k(retrieved, relevant, k))
        recall_lists[k].append(compute_recall_at_k(retrieved, relevant, k))

    detail_rows.append({
        'query_posting_id': q_posting,
        'label_group'     : df.at[q_idx, 'label_group'],
        'ap@5'            : ap5,
        'retrieved_top5'  : ','.join(retrieved[:5]),
        'n_relevant'      : len(relevant),
    })

# ── In kết quả ────────────────────────────────────────────
mAP5 = np.mean(map5_list)
print("\n" + "=" * 60)
print("  📈 KẾT QUẢ ĐÁNH GIÁ – Multimodal Late Fusion")
print("  (EfficientNet-B4 + TF-IDF | FAISS IndexFlatIP)")
print("=" * 60)
print(f"  mAP@5         : {mAP5:.4f}")
print("-" * 60)
header = f"  {'Metric':<20}" + "".join(f"@{k:<8}" for k in K_LIST)
print(header)
print("-" * 60)
print(f"  {'Precision':<20}" + "".join(f"{np.mean(precision_lists[k]):.4f}  " for k in K_LIST))
print(f"  {'Recall':<20}" + "".join(f"{np.mean(recall_lists[k]):.4f}  " for k in K_LIST))
print("=" * 60)

# ── Lưu file CSV ─────────────────────────────────────────
# 1. Metrics tổng hợp
metrics_data = {'mAP@5': [mAP5]}
for k in K_LIST:
    metrics_data[f'Precision@{k}'] = [np.mean(precision_lists[k])]
    metrics_data[f'Recall@{k}']    = [np.mean(recall_lists[k])]

metrics_df = pd.DataFrame(metrics_data)
metrics_csv = os.path.join(RESULTS, 'metrics_fusion_eff_tfidf.csv')
metrics_df.to_csv(metrics_csv, index=False)
print(f"\n   💾 Metrics lưu: {metrics_csv}")

# 2. Chi tiết từng query
detail_df = pd.DataFrame(detail_rows)
detail_csv = os.path.join(RESULTS, 'fusion_detail_eff_tfidf.csv')
detail_df.to_csv(detail_csv, index=False)
print(f"   💾 Detail lưu : {detail_csv}")

In [ ]:
# ============================================================
# CELL 8: VẼ BIỂU ĐỒ PHÂN TÍCH LỖI (ERROR ANALYSIS)
# ============================================================
plt.rcParams.update({'font.size': 9, 'axes.titlesize': 8})


def load_image_safe(path: str, size: tuple = (224, 224)) -> np.ndarray:
    """Đọc ảnh an toàn, trả về numpy array RGB."""
    try:
        img = Image.open(path).convert('RGB').resize(size)
        return np.array(img)
    except Exception:
        return np.zeros((*size, 3), dtype=np.uint8)


def visualize_query_result(
    q_idx: int,
    retrieved_ids: list,
    df: pd.DataFrame,
    posting2idx: dict,
    ax_row,
    title_prefix: str = ''
):
    """
    Vẽ 1 hàng: [Query] + [Top-5 Retrieved].
    Viền xanh = TP, viền đỏ = FP.
    """
    q_posting   = idx2posting[q_idx]
    q_label     = df.at[q_idx, 'label_group']
    q_title     = df.at[q_idx, 'title']
    q_img_path  = df.at[q_idx, 'image_path']
    relevant     = gt_map[q_posting]

    all_items = [(q_posting, q_img_path, q_label, q_title, None)] + [
        (
            pid,
            df.at[posting2idx[pid], 'image_path'],
            df.at[posting2idx[pid], 'label_group'],
            df.at[posting2idx[pid], 'title'],
            pid in relevant,
        )
        for pid in retrieved_ids[:5]
        if pid in posting2idx
    ]

    for col_idx, (pid, img_path, label, title_text, is_tp) in enumerate(all_items):
        ax = ax_row[col_idx]
        img_arr = load_image_safe(img_path)
        ax.imshow(img_arr)
        ax.set_xticks([])
        ax.set_yticks([])

        if col_idx == 0:
            # Query
            ax.set_title(f'{title_prefix}QUERY\n{title_text[:30]}…', color='royalblue', fontweight='bold')
            for spine in ax.spines.values():
                spine.set_edgecolor('royalblue')
                spine.set_linewidth(3)
        else:
            rank  = col_idx
            color = '#2ecc71' if is_tp else '#e74c3c'   # xanh lá = TP, đỏ = FP
            label_flag = '✅TP' if is_tp else '❌FP'
            ax.set_title(f'Top-{rank} {label_flag}\n{title_text[:30]}…', color=color, fontweight='bold')
            for spine in ax.spines.values():
                spine.set_edgecolor(color)
                spine.set_linewidth(3)


def visualize_and_save(
    query_indices: list,
    I: np.ndarray,
    df: pd.DataFrame,
    posting2idx: dict,
    filename: str,
    suptitle: str
):
    """Vẽ và lưu figure cho danh sách query."""
    n_rows  = len(query_indices)
    n_cols  = 6   # 1 Query + 5 Top-5
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 3))
    if n_rows == 1:
        axes = axes[np.newaxis, :]

    fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.01)

    for row, q_idx in enumerate(query_indices):
        q_posting = idx2posting[q_idx]
        neighbor_indices = I[q_idx].tolist()
        retrieved = [
            idx2posting[ni]
            for ni in neighbor_indices
            if idx2posting[ni] != q_posting
        ][:5]
        visualize_query_result(
            q_idx, retrieved, df, posting2idx,
            axes[row], title_prefix=f'[Q{row+1}] '
        )

    plt.tight_layout()
    save_path = os.path.join(RESULTS, filename)
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"   💾 Đã lưu: {save_path}")


# ── Tự động tìm 2 mẫu True Positive + 2 mẫu False Positive ─
print("🔍 Quét các mẫu điển hình...")
tp_samples = []   # Query có Top-1 đúng
fp_samples = []   # Query có Top-1 sai

for q_idx in tqdm(range(N), desc='Scanning samples', unit='q'):
    if len(tp_samples) >= 2 and len(fp_samples) >= 2:
        break
    q_posting = idx2posting[q_idx]
    relevant  = gt_map[q_posting]
    if not relevant:
        continue

    neighbor_indices = I[q_idx].tolist()
    retrieved = [
        idx2posting[ni]
        for ni in neighbor_indices
        if idx2posting[ni] != q_posting
    ][:5]

    if not retrieved:
        continue

    # Kiểm tra: Top-1 có đúng không?
    top1_correct = (retrieved[0] in relevant)
    # Kiểm tra: Tất cả Top-5 đều đúng (True Positive hoàn toàn)
    all_top5_tp = all(r in relevant for r in retrieved[:min(5, len(relevant))])

    if all_top5_tp and len(tp_samples) < 2:
        tp_samples.append(q_idx)
    elif not top1_correct and len(fp_samples) < 2:
        fp_samples.append(q_idx)

print(f"   ✅ True Positive samples : {tp_samples}")
print(f"   ❌ False Positive samples: {fp_samples}")

# ── Vẽ và lưu ──────────────────────────────────────────────
print("\n🎨 Vẽ biểu đồ True Positive (đúng hoàn toàn)...")
visualize_and_save(
    tp_samples, I, df, posting2idx,
    filename='error_analysis_true_positives.png',
    suptitle='✅ True Positive – Query đúng hoàn toàn (EfficientNet-B4 + TF-IDF)'
)

print("\n🎨 Vẽ biểu đồ False Positive (Top-1 sai)...")
visualize_and_save(
    fp_samples, I, df, posting2idx,
    filename='error_analysis_false_positives.png',
    suptitle='❌ False Positive – Query bị sai ở Top-1 (EfficientNet-B4 + TF-IDF)'
)

# 📝 CELL 9: Nhận xét Chuyên sâu – Multimodal Late Fusion

---

## 🔷 1. Sự vượt trội của EfficientNet-B4 trong Fine-Grained Recognition

EfficientNet-B4 được huấn luyện trên ImageNet với hơn 1.28 triệu ảnh, cho phép nó học được các đặc trưng hình ảnh phân cấp từ thấp (cạnh, màu sắc, kết cấu) đến cao (hình dạng đối tượng, phong cách sản phẩm). Trong bài toán **fine-grained visual search** trên Shopee – nơi hai sản phẩm cùng loại nhưng khác màu/kích thước cần được phân biệt – EfficientNet-B4 với resolution đầu vào **380×380** và kiến trúc compound scaling vượt trội hơn ResNet-50 hay MobileNet ở cùng số lượng tham số. Vector 1792 chiều từ lớp `avgpool` mã hóa đặc trưng ngữ nghĩa thị giác phong phú, giúp phân biệt sản phẩm có ngoại hình tương tự nhau.

---

## 🔷 2. Thực dụng của TF-IDF trước Rào cản Đa ngôn ngữ và Spam từ khóa Shopee

Dữ liệu tiêu đề sản phẩm Shopee cực kỳ đa dạng: pha trộn tiếng Anh, tiếng Thái, tiếng Indonesia, tiếng Việt, lẫn các ký tự đặc biệt và từ khóa spam ("FREE SHIP", "Murah", "terjamin"). TF-IDF với `max_features=3000` tỏ ra **thực dụng** hơn Word Embeddings đơn ngữ: (a) không cần pre-training đa ngôn ngữ tốn kém, (b) tự nhiên giảm trọng số từ xuất hiện quá phổ biến ("free", "sale") qua IDF, (c) bắt được từ đặc trưng sản phẩm (brand name, model number). Hàm `clean_text` bỏ ký tự đặc biệt nhưng giữ lại ký tự Unicode giúp xử lý đa ngôn ngữ mà không cần tokenizer chuyên dụng.

---

## 🔷 3. Lợi ích của FAISS trong Tăng tốc Quét Siêu vector 4792 chiều

Với **N ≈ 34.250 ảnh**, tìm kiếm brute-force cosine similarity yêu cầu tính **34.250 × 34.250 ≈ 1.17 tỷ** phép nhân vector (mỗi vector 4792 chiều). FAISS `IndexFlatIP` tận dụng BLAS/cuBLAS để thực hiện batch matrix multiplication trên GPU, giảm thời gian search từ hàng chục phút xuống còn **vài giây**. Đặc biệt, việc L2-normalize siêu vector trước giúp Inner Product trở thành phép đo Cosine Similarity chính xác, loại bỏ cần thiết phải dùng `IndexFlatL2` hay normalize tại query time, tối ưu bộ nhớ và tốc độ.

---

## 🔷 4. Phân tích Nguyên nhân False Positives Còn Sót Lại

Qua phân tích lỗi hình ảnh, các False Positive chủ yếu do:
- **Visual confounders**: Sản phẩm khác nhóm nhưng có hình dạng/màu sắc giống nhau (ví dụ: hai loại áo phông trắng khác brand). EfficientNet chỉ thấy đặc trưng hình thức, chưa hiểu ngữ nghĩa sản phẩm.
- **Text keyword overlap**: Hai sản phẩm khác nhau chia sẻ nhiều từ khóa giống nhau ("cotton", "size M", "100% original") khiến TF-IDF vector tương tự nhau.
- **Ảnh chất lượng thấp**: Ảnh bị blur hoặc background phức tạp khiến EfficientNet tập trung vào background thay vì sản phẩm.
- **Unbalanced modality weight**: Late Fusion đơn giản (concatenate với trọng số bằng nhau) chưa điều chỉnh được trường hợp một modality tin cậy hơn modality kia.

---

## 🔷 5. Hướng Tinh chỉnh Bằng ArcFace cho Tuần 5

Để vượt qua giới hạn của Late Fusion với pretrained features, **Tuần 5** sẽ áp dụng **ArcFace loss** (Additive Angular Margin Loss) để fine-tune EfficientNet-B4 trực tiếp trên tập Shopee:
- **ArcFace** ép buộc embedding của cùng `label_group` thu hẹp lại (intra-class compactness) và tăng khoảng cách giữa các nhóm khác nhau (inter-class discrepancy) trên không gian hypersphere – lý tưởng cho visual search.
- Kết hợp với **hard negative mining** (chọn mẫu âm khó nhất trong batch) để tập trung vào các trường hợp model hiện tại còn nhầm lẫn.
- **Roadmap**: EfficientNet-B4 + ArcFace → extract features → Late Fusion với TF-IDF (hoặc sentence-BERT) → FAISS → dự kiến mAP@5 cải thiện **10-20%** so với kết quả Tuần 4.